In [ ]:
# Generate distance matrices ASPD/ASPH

import ast
import glob
import logging
import os
import sys
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import wntr

logging.basicConfig(level=logging.WARNING, format="%(asctime)s - %(levelname)s - %(message)s", stream=sys.stdout)
logger = logging.getLogger()

network_name = "Hanoi"
network_inp_file = "networks/Hanoi.inp"

base_output_dir = Path("output")
distance_matrix_dir = base_output_dir / "Distance_Matrices_Length_Weighted"
results_dirs = [base_output_dir / "TBPO", base_output_dir / "CSPO"]


def parse_sensor_ids(value: object) -> set[str]:
    if pd.isna(value):
        return set()
    if isinstance(value, (set, list, tuple)):
        return {str(s).strip() for s in value if str(s).strip()}
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return set()
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                return {str(item).strip() for item in parsed if str(item).strip()}
            except (ValueError, SyntaxError):
                s = s[1:-1].strip()
                if not s:
                    return set()
                return {
                    item.strip().strip("'").strip('"')
                    for item in s.split(",")
                    if item.strip().strip("'").strip('"')
                }
        for delim in [",", ";"]:
            if delim in s:
                return {item.strip() for item in s.split(delim) if item.strip()}
        parts = [p.strip() for p in s.split()]
        return {p for p in parts if p}
    logger.warning(f"Could not parse sensor IDs: {value!r}")
    return set()


def calculate_aspd(selected_idx: list[int], length_weighted_matrix: np.ndarray) -> float:
    if len(selected_idx) < 2:
        return 0.0
    try:
        sub = length_weighted_matrix[np.ix_(selected_idx, selected_idx)].astype(float)
        np.fill_diagonal(sub, np.inf)
        min_distances = np.min(sub, axis=1)
        finite = min_distances[np.isfinite(min_distances)]
        if len(finite) > 0:
            return float(np.mean(finite))
        logger.warning("No finite distances found for ASPD")
        return 0.0
    except IndexError:
        logger.warning("IndexError in ASPD calculation (indices out of bounds)")
        return -1.0
    except Exception as e:
        logger.error(f"Unexpected error in ASPD calculation: {e}")
        return -1.0


def calculate_asph(selected_idx: list[int], unweighted_matrix: np.ndarray) -> float:
    if len(selected_idx) < 2:
        return 0.0
    try:
        sub = unweighted_matrix[np.ix_(selected_idx, selected_idx)].astype(float)
        np.fill_diagonal(sub, np.inf)
        min_distances = np.min(sub, axis=1)
        finite = min_distances[np.isfinite(min_distances)]
        if len(finite) > 0:
            return float(np.mean(finite))
        logger.warning("No finite distances found for ASPH")
        return 0.0
    except IndexError:
        logger.warning("IndexError in ASPH calculation (indices out of bounds)")
        return -1.0
    except Exception as e:
        logger.error(f"Unexpected error in ASPH calculation: {e}")
        return -1.0


distance_matrix_dir.mkdir(parents=True, exist_ok=True)
length_weighted_parquet = distance_matrix_dir / f"{network_name}_distance_matrix_length_weighted.parquet"
unweighted_parquet = distance_matrix_dir / f"{network_name}_distance_matrix_unweighted.parquet"

length_weighted_df: pd.DataFrame | None = None
unweighted_df: pd.DataFrame | None = None

if length_weighted_parquet.exists():
    length_weighted_df = pd.read_parquet(length_weighted_parquet)
else:
    try:
        wn = wntr.network.WaterNetworkModel(network_inp_file)
        G = wn.to_graph()
        for _, pipe in wn.pipes():
            n1, n2 = pipe.start_node_name, pipe.end_node_name
            if G.has_edge(n1, n2):
                for key in G[n1][n2]:
                    G[n1][n2][key]["length"] = pipe.length
            else:
                logger.warning(f"Edge for pipe between {n1} and {n2} not found in graph.")
        sample_edge = list(G.edges(data=True))[0]
        if "length" not in sample_edge[2]:
            raise ValueError("Failed to set 'length' attribute on graph edges")
        path_lengths = dict(nx.all_pairs_dijkstra_path_length(G.to_undirected(), weight="length"))
        node_order = sorted(G.nodes())
        length_weighted_df = pd.DataFrame(path_lengths, index=node_order, columns=node_order).fillna(np.inf)
        length_weighted_df.to_parquet(length_weighted_parquet, engine="pyarrow", compression="snappy")
    except Exception as e:
        logger.error(f"Failed to generate length-weighted matrix for {network_name}: {e}")
        length_weighted_df = None

if unweighted_parquet.exists():
    unweighted_df = pd.read_parquet(unweighted_parquet)
else:
    try:
        wn = wntr.network.WaterNetworkModel(network_inp_file)
        G = wn.to_graph()
        path_lengths_unweighted = dict(nx.all_pairs_shortest_path_length(G.to_undirected()))
        node_order = sorted(G.nodes())
        unweighted_df = pd.DataFrame(path_lengths_unweighted, index=node_order, columns=node_order).fillna(np.inf)
        unweighted_df.to_parquet(unweighted_parquet, engine="pyarrow", compression="snappy")
    except Exception as e:
        logger.error(f"Failed to generate unweighted matrix for {network_name}: {e}")
        unweighted_df = None

if length_weighted_df is None or unweighted_df is None:
    raise RuntimeError("Cannot proceed with ASPD/ASPH recalculation")

length_weighted_df.index = length_weighted_df.index.astype(str)
length_weighted_df.columns = length_weighted_df.columns.astype(str)
unweighted_df.index = unweighted_df.index.astype(str)
unweighted_df.columns = unweighted_df.columns.astype(str)

node_to_idx = {name: i for i, name in enumerate(length_weighted_df.index)}
length_weighted_matrix = length_weighted_df.values
unweighted_matrix = unweighted_df.values

result_files: list[str] = []
for results_dir in results_dirs:
    if not (results_dir.exists() and results_dir.is_dir()):
        logger.warning(f"Results directory not found: {results_dir}")
        continue
    for f in glob.glob(os.path.join(results_dir, "*_pareto_optimal_solutions_summary.csv")):
        if "_with_ASPD" in Path(f).name:
            continue
        result_files.append(f)

if not result_files:
    print(f"No result files found for network '{network_name}' in TBPO/CSPO.")
else:
    processed_files = 0
    for file_path in result_files:
        file_path_obj = Path(file_path)
        try:
            df = pd.read_csv(file_path)
            sensor_col = next((c for c in ["Selected_Sensor_IDs", "Original_Sensor_IDs_Str"] if c in df.columns), None)
            if not sensor_col:
                raise ValueError(f"Could not find a sensor ID column in {file_path_obj.name}")
            first_valid = next((v for v in df[sensor_col].tolist() if not pd.isna(v)), None)
            if first_valid is not None:
                sensor_ids_first = parse_sensor_ids(first_valid)
                if sensor_ids_first:
                    mapped = [sid for sid in sensor_ids_first if sid in node_to_idx]
                    if len(mapped) == 0:
                        logger.warning(f"Skipping {file_path_obj.name}: sensor IDs do not match '{network_name}'.")
                        continue
            aspd_values: list[float] = []
            asph_values: list[float] = []
            for _, row in df.iterrows():
                sensor_ids = parse_sensor_ids(row[sensor_col])
                selected_idx = [node_to_idx[sid] for sid in sensor_ids if sid in node_to_idx]
                aspd_values.append(calculate_aspd(selected_idx, length_weighted_matrix))
                asph_values.append(calculate_asph(selected_idx, unweighted_matrix))
            df["Average_Shortest_Path_Distance_m"] = aspd_values
            df["Average_Shortest_Path_Hops"] = asph_values
            output_subdir = file_path_obj.parent / f"{file_path_obj.parent.name}_with_ASPD_ASPH"
            output_subdir.mkdir(exist_ok=True)
            output_csv_path = output_subdir / f"{file_path_obj.stem}_with_ASPD_ASPH.csv"
            df.to_csv(
                output_csv_path,
                index=False,
                float_format="%.3f",
                encoding="utf-8-sig",
                lineterminator="\n",
            )
            processed_files += 1
        except Exception as e:
            logger.error(f"Failed to process file {file_path}: {e}")
    print(
        f"Recalculated ASPD/ASPH for {processed_files} files. "
        "Outputs saved under '*_with_ASPD_ASPH/' next to each results folder."
    )